# data-conduit Tutorial & Demo (Real JPVA Dataset)

**End-to-end walkthrough using `JPVA191B_23102025_run1_g0` real data.**

This notebook now uses the real experiment directory in this repository:
- `JPVA191B_23102025_run1_g0/bonsai` (HARP + CSV behavior streams)
- `JPVA191B_23102025_run1_g0/NPX` (Neuropixels arrays including sync channel)
- `JPVA191B_23102025_run1_g0/DLC` (pose tracking outputs)

---

### Table of Contents

| # | Section | Module(s) |
|---|---------|-----------|
| 1 | [IO — File Discovery & Reading](#1-io) | `io.collect_dfs`, `io.readers` |
| 2 | [DataSource & Subclasses](#2-datasource) | `datasource.DataSource`, `Device`, `FileTypeData` + presets |
| 3 | [MultiSource & MultiDevice](#3-multisource) | `multisource.MultiSource`, `MultiDevice`, `Nosepoke` |
| 4 | [Virtual Arrays & Queries](#4-virtualarrays) | `virtualarrays.construct_data_array`, `construct_lookup_array`, `ulookup` |
| 5 | [TTL Synchronisation](#5-ttlsync) | `ttlsync.extract_ttl_segments`, `TTLSyncModel` |
| 6 | [Global Timebase Mapping](#6-globaltimes) | `globaltimes.create_global_clock`, `index_map_util` |
| 7 | [Segmentation Utilities](#7-segment) | `segment.segment_boolean_series`, `slice_event_windows`, `slice_dataarray_windows` |
| 8 | [Timestamps & Utilities](#8-utilities) | `timestamps`, `harptools`, `utils` |
| 9 | [API Quick-Reference](#9-api) | Full function index |

In [12]:
# ── Shared imports & explicit path arguments ──────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# Configure these arguments once.
base_path = Path("./JPVA191B_23102025_run1_g0")
experiment_directory_path = base_path
bonsai_directory_path = base_path / "bonsai"
npx_directory_path = base_path / "NPX"
dlc_directory_path = base_path / "DLC"

# Explicit non-experiment paths used elsewhere in the demo.
harp_device_yaml_path = Path("/home/callum/Keshavarzi-Lab-Workspace/data-conduit/device.yml")
misc_directory_path = Path("/home/callum/Keshavarzi-Lab-Workspace/data-conduit/Misc")

# if not bonsai_directory_path.is_dir():
#     raise NotADirectoryError(f"The specified base path is not a directory: {bonsai_directory_path}")

print(f"base_path:                 {base_path}")
print(f"experiment_directory_path: {experiment_directory_path}")
print(f"bonsai_directory_path:     {bonsai_directory_path}")
print(f"npx_directory_path:        {npx_directory_path}")
print(f"dlc_directory_path:        {dlc_directory_path}")
print(f"harp_device_yaml_path:     {harp_device_yaml_path}")

base_path:                 JPVA191B_23102025_run1_g0
experiment_directory_path: JPVA191B_23102025_run1_g0
bonsai_directory_path:     JPVA191B_23102025_run1_g0/bonsai
npx_directory_path:        JPVA191B_23102025_run1_g0/NPX
dlc_directory_path:        JPVA191B_23102025_run1_g0/DLC
harp_device_yaml_path:     /home/callum/Keshavarzi-Lab-Workspace/data-conduit/device.yml


In [13]:
# ── Quick inventory of the real JPVA dataset ─────────────────────────────────
def show_tree(root: Path, max_depth: int, max_entries: int):
    n = 0
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        indent = "  " * (len(rel.parts) - 1)
        marker = "[D]" if p.is_dir() else "[F]"
        print(f"{indent}{marker} {rel}")
        n += 1
        if n >= max_entries:
            print("... truncated ...")
            break

print("\nBonsai (depth<=2):")
show_tree(bonsai_directory_path, max_depth=2, max_entries=120)

print("\nNPX files:")
for p in sorted(npx_directory_path.glob("*")):
    print("[F]", p.name)

print("\nDLC files:")
for p in sorted(dlc_directory_path.glob("*")):
    print("[F]", p.name)


Bonsai (depth<=2):
[D] Behavior0
  [F] Behavior0/Behavior0_34_1904-01-01T02-00-00.bin
  [F] Behavior0/Behavior0_34_1904-01-01T02-00-00.bin:Zone.Identifier
  [F] Behavior0/Behavior0_34_1904-01-01T02-00-00.bin:Zone.Identifier:Zone.Identifier
  [F] Behavior0/Behavior0_35_1904-01-01T02-00-00.bin
  [F] Behavior0/Behavior0_35_1904-01-01T02-00-00.bin:Zone.Identifier
  [F] Behavior0/Behavior0_35_1904-01-01T02-00-00.bin:Zone.Identifier:Zone.Identifier
  [F] Behavior0/Behavior0_44_1904-01-01T01-00-00.bin
  [F] Behavior0/Behavior0_44_1904-01-01T01-00-00.bin:Zone.Identifier
  [F] Behavior0/Behavior0_44_1904-01-01T01-00-00.bin:Zone.Identifier:Zone.Identifier
  [F] Behavior0/Behavior0_44_1904-01-01T02-00-00.bin
  [F] Behavior0/Behavior0_44_1904-01-01T02-00-00.bin:Zone.Identifier
  [F] Behavior0/Behavior0_44_1904-01-01T02-00-00.bin:Zone.Identifier:Zone.Identifier
  [F] Behavior0/Behavior0_78_1904-01-01T01-00-00.bin
  [F] Behavior0/Behavior0_78_1904-01-01T01-00-00.bin:Zone.Identifier
  [F] Behavior0/

---
<a id="1-io"></a>
## 1  IO — File Discovery & Reading

The `io` layer is used here directly on real `JPVA191B_23102025_run1_g0` files.

Key concepts:
- **readers**: functions that map `(path, **kwargs) -> DataFrame`
- **level selectors**: filter folders during directory walking
- **flatten**: collapse nested dict keys when needed

In this section we load:
- Behavior HARP binary folders (`Behavior0` ... `Behavior5`)
- CSV folders (`ExperimentEvents`, `VideoData`, `VisualEnvironment`)

In [14]:
from data_conduit.io import collect_dfs, read_csv, collect_folders

# ── 1a. collect_folders on bonsai root ───────────────────────────────────────
folders = collect_folders(bonsai_directory_path)
print("Bonsai top-level folders:")
print([f.name for f in folders])

Bonsai top-level folders:
['Behavior0', 'Behavior1', 'Behavior2', 'Behavior3', 'Behavior4', 'Behavior5', 'ExperimentEvents', 'InnerRotation', 'NosepokeRotation', 'OuterRotation', 'SessionSettings', 'SoundCard', 'VideoData', 'VisualEnvironment']


In [15]:
# ── 1b. collect_dfs on real CSV streams ──────────────────────────────────────
csv_dfs = collect_dfs(
    base_path=folders[0].parent,  # Use parent of first folder (bonsai dir)
    readers={".csv": read_csv},
    l0_selector=["ExperimentEvents", "VideoData", "VisualEnvironment"],
    verbose=True,
    keep_empty=False,
)

print("\nTop-level keys:", list(csv_dfs.keys()))
for topk in csv_dfs:
    sub = list(csv_dfs[topk].keys()) if isinstance(csv_dfs[topk], dict) else type(csv_dfs[topk]).__name__
    print(f"  {topk}: {sub}")

example_events_key = next(iter(csv_dfs["ExperimentEvents"]))
csv_dfs["ExperimentEvents"][example_events_key].head()

Failed to read JPVA191B_23102025_run1_g0/bonsai/VisualEnvironment/VisualEnvironment_1904-01-01T02-00-00.csv with reader for .csv: Expected 2 fields in line 3, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

Top-level keys: ['ExperimentEvents', 'VideoData']
  ExperimentEvents: ['ExperimentEvents_1904-01-01T01-00-00', 'ExperimentEvents_1904-01-01T02-00-00']
  VideoData: ['VideoData_1904-01-01T01-00-00', 'VideoData_1904-01-01T02-00-00']


,Value
Seconds,
6240.664,Request camera start


In [16]:
# ── 1c. Level selectors on real HARP behavior folders ───────────────────────
from data_conduit.utils import starts_with

behavior_bins = collect_dfs(
    base_path=bonsai_directory_path,
    readers=None,
    l0_selector=starts_with("Behavior"),
    flatten=False,
    keep_empty=False,
    verbose=False,
)

print("Behavior folders discovered:", list(behavior_bins.keys())[:6])
print("Example file stems in Behavior0:")
print(list(behavior_bins.get("Behavior0", {}).keys())[:8])

Behavior folders discovered: ['Behavior0', 'Behavior1', 'Behavior2', 'Behavior3', 'Behavior4', 'Behavior5']
Example file stems in Behavior0:
['Behavior0_34_1904-01-01T02-00-00', 'Behavior0_34_1904-01-01T02-00-00.bin:Zone', 'Behavior0_34_1904-01-01T02-00-00.bin:Zone.Identifier:Zone', 'Behavior0_35_1904-01-01T02-00-00', 'Behavior0_35_1904-01-01T02-00-00.bin:Zone', 'Behavior0_35_1904-01-01T02-00-00.bin:Zone.Identifier:Zone', 'Behavior0_44_1904-01-01T01-00-00', 'Behavior0_44_1904-01-01T01-00-00.bin:Zone']


In [17]:
# ── 1d. Flatten CSV view for quick indexing ──────────────────────────────────
csv_flat = collect_dfs(
    base_path=bonsai_directory_path,
    readers={".csv": read_csv},
    l0_selector=["ExperimentEvents", "InnerRotation", "OuterRotation", "NosepokeRotation"],
    flatten=True,
    separator="/",
    verbose=False,
)

print("Flat keys sample:")
for k in list(csv_flat.keys())[:10]:
    print(" ", k)

Flat keys sample:
  ExperimentEvents/ExperimentEvents_1904-01-01T01-00-00
  ExperimentEvents/ExperimentEvents_1904-01-01T02-00-00
  InnerRotation/InnerRotation_1904-01-01T01-00-00
  InnerRotation/InnerRotation_1904-01-01T02-00-00
  NosepokeRotation/NosepokeRotation_1904-01-01T01-00-00
  NosepokeRotation/NosepokeRotation_1904-01-01T02-00-00
  OuterRotation/OuterRotation_1904-01-01T01-00-00
  OuterRotation/OuterRotation_1904-01-01T02-00-00


In [18]:
# ── 1e. Custom reader example on real SessionSettings JSONL ──────────────────
import json
from data_conduit.io import add_reader

def read_jsonl_records(path, **kwargs):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.json_normalize(rows)

add_reader("jsonl_records", read_jsonl_records)
print("Custom reader 'jsonl_records' registered.")

ValueError: A reader with the name 'jsonl_records' is already registered.

---
<a id="2-datasource"></a>
## 2  DataSource & Subclasses (Real HARP Data)

This section uses the real Bonsai and HARP files from JPVA.

Hierarchy:
- `DataSource`: generic folder->dict loader
- `Device`: HARP `.bin` device/register loader
- `FileTypeData`: CSV/JSON/YAML loader presets

In [19]:
# ── 2a. DataSource on real CSV folders ───────────────────────────────────────
from data_conduit.datasource import DataSource

ds_csv = DataSource(
    experiment_directory_path=bonsai_directory_path,
    readers={".csv": read_csv},
    l0_selector=["ExperimentEvents", "VideoData"],
    verbose=True,
)

print("dfs_dict keys:", list(ds_csv.dfs_dict.keys()))
events_key = next(iter(ds_csv.dfs_dict["ExperimentEvents"]))
print("ExperimentEvents sample key:", events_key)
ds_csv.dfs_dict["ExperimentEvents"][events_key].head()

dfs_dict keys: ['ExperimentEvents', 'VideoData']
ExperimentEvents sample key: ExperimentEvents_1904-01-01T01-00-00


,Value
Seconds,
6240.664,Request camera start


In [21]:
# ── 2b. Device class on real HARP behavior binaries ──────────────────────────
from data_conduit.datasource import Device

behavior_device = Device(
    experiment_directory_path=bonsai_directory_path,
    harp_device_yaml_path=harp_device_yaml_path,
    device_type="Behavior",
    device_list=[f"Behavior{i}" for i in range(6)],
    matching="T02-00-00",
    verbose=True,
)

print("Loaded behavior devices:", list(behavior_device.dfs_dict.keys())[:6])
print("Behavior0 registers:", list(behavior_device.dfs_dict.get("Behavior0", {}).keys())[:8])

ValueError: 
                             Invalid keyword argument: 'matching'.
                             Selectors must follow 'l{n}_selector' pattern, where n is a non-negative integer 
                             (e.g., 'l0_selector', 'l1_selector').
                

In [22]:
# ── 2c. FileTypeData preset: ExperimentEvents ────────────────────────────────
from data_conduit.datasource import ExperimentEvents

events_src = ExperimentEvents(
    experiment_directory_path=bonsai_directory_path,
    verbose=True,
)

print("ExperimentEvents folders:", list(events_src.dfs_dict.keys()))
event_folder = next(iter(events_src.dfs_dict))
event_file = next(iter(events_src.dfs_dict[event_folder]))
events_src.dfs_dict[event_folder][event_file].head()

ExperimentEvents folders: ['ExperimentEvents']


,Event
Time,
6240.664,Request camera start


> The JPVA demo now uses real `Device`/`FileTypeData` loaders directly.

- `Device` reads HARP `.bin` streams using `device.yml`
- `ExperimentEvents` reads real event CSV files
- both produce `dfs_dict` structures compatible with `MultiSource`

---
<a id="3-multisource"></a>
## 3  MultiSource & MultiDevice on JPVA

`MultiDevice` loads HARP device trees, then `MultiSource` builds aligned data arrays via virtual maps.

In this section we instantiate the `Nosepoke` preset on real `Behavior0..Behavior5` data and inspect produced arrays.

In [23]:
# ── 3a. Nosepoke preset on real JPVA HARP data ───────────────────────────────
from data_conduit.multisource.multidevice import Nosepoke

nosepoke = Nosepoke(
    experiment_directory_path=bonsai_directory_path,
    harp_device_yaml_path=harp_device_yaml_path,
    device_type="Behavior",
    device_list=[f"Behavior{i}" for i in range(6)],
    verbose=True,
)

print("data_arrays keys:", list(nosepoke.data_arrays.keys()))
print("lookup_arrays keys:", list(nosepoke.lookup_arrays.keys()))

primary_key = list(nosepoke.data_arrays.keys())[0]
print("Primary key:", primary_key)
nosepoke.data_arrays[primary_key]

Failed to read JPVA191B_23102025_run1_g0/bonsai/Behavior4/Behavior4_1904-01-01T01-00-00.bin with reader for .bin: invalid literal for int() with base 10: '1904-01-01T01-00-00'
Failed to read JPVA191B_23102025_run1_g0/bonsai/Behavior5/Behavior5_1904-01-01T01-00-00.bin with reader for .bin: invalid literal for int() with base 10: '1904-01-01T01-00-00'


ValueError: Rekeying would overwrite data for device 'Behavior0' at register '44'.

In [24]:
# ── 3b. Inspect lookup metadata for real channels ─────────────────────────────
primary_key = list(nosepoke.lookup_arrays.keys())[0]
lookup = nosepoke.lookup_arrays[primary_key]

print("Lookup dims:", lookup.dims)
print("Lookup coord names:", list(lookup.coords))

global_dim = lookup.dims[0]
print(f"\nFirst 8 {global_dim} entries:")
for ch in lookup.coords[global_dim].values[:8]:
    row = lookup.sel({global_dim: ch})
    coords = {c: row.coords[c].item() for c in row.coords if c != global_dim}
    print(ch, "->", coords)

NameError: name 'nosepoke' is not defined

In [25]:
# ── 3c. test_values mode for map debugging (same real mapping) ───────────────
nosepoke_test = Nosepoke(
    experiment_directory_path=bonsai_directory_path,
    harp_device_yaml_path=harp_device_yaml_path,
    device_type="Behavior",
    device_list=[f"Behavior{i}" for i in range(6)],
    test_values=True,
    verbose=False,
)

k = list(nosepoke_test.data_arrays.keys())[0]
da_test = nosepoke_test.data_arrays[k]
global_dim = da_test.dims[1] if len(da_test.dims) > 1 else da_test.dims[0]
print(f"Key: {k}, dims: {da_test.dims}")
print("First timestep labels:")
for ch in da_test.coords[global_dim].values[:8]:
    print(" ", da_test.sel({global_dim: ch}).values[0])

ValueError: Rekeying would overwrite data for device 'Behavior0' at register '44'.

> `Nosepoke` is now demonstrated on the real JPVA run, not synthetic arrays.

It auto-builds channel maps for the default behavior devices and outputs:
- one data array per data key (e.g. Activations, LEDs, Valves, Rewards)
- matching lookup arrays for virtual-coordinate queries

---
<a id="4-virtualarrays"></a>
## 4  Virtual Arrays & Queries (Real Nosepoke Outputs)

Here we take arrays already produced from real JPVA behavior devices and query them with lookup metadata.

In [26]:
# ── 4a. Pull one real data/lookup pair from Nosepoke ─────────────────────────
from data_conduit.virtualarrays import ulookup

data_key = list(nosepoke.data_arrays.keys())[0]
da_filled = nosepoke.data_arrays[data_key]
lookup = nosepoke.lookup_arrays[data_key]

print("data_key:", data_key)
print("data dims:", da_filled.dims, "shape:", da_filled.shape)
print("lookup dims:", lookup.dims)
da_filled

NameError: name 'nosepoke' is not defined

In [27]:
# ── 4b. Query channels by virtual coords using ulookup ───────────────────────
global_dim = lookup.dims[0]

behavior0_channels = ulookup(
    lookup,
    global_coord_name=global_dim,
    device="Behavior0",
)

print("Behavior0 channels:", behavior0_channels[:10])
print("Count:", len(behavior0_channels))

NameError: name 'lookup' is not defined

In [28]:
# ── 4c. Query by register and localID ────────────────────────────────────────
reg32_channels = ulookup(lookup, global_coord_name=global_dim, register="32")
dip0_channels = ulookup(lookup, global_coord_name=global_dim, localID="DIPort0")

print("Register 32 channels:", reg32_channels[:10], "... total", len(reg32_channels))
print("DIPort0 channels:", dip0_channels[:10], "... total", len(dip0_channels))

NameError: name 'lookup' is not defined

In [29]:
# ── 4d. Slice real data array with queried channels ───────────────────────────
subset_channels = behavior0_channels[:3] if len(behavior0_channels) >= 3 else behavior0_channels
da_subset = da_filled.sel({global_dim: subset_channels})

print("Subset channels:", subset_channels)
print("Subset shape:", da_subset.shape)
da_subset

NameError: name 'behavior0_channels' is not defined

In [30]:
# ── 4e. DataArray accessor shorthand: .ulookup() ─────────────────────────────
from data_conduit.virtualarrays import LookupAccessorConstructor  # noqa: F401

# Same query via accessor on the real DataArray
b1_channels = da_filled.ulookup(device="Behavior1")
print("Behavior1 via accessor:", b1_channels[:10], "... total", len(b1_channels))

reg34_channels = da_filled.ulookup.sel(register="34")
print("Register 34 via accessor:", reg34_channels[:10], "... total", len(reg34_channels))

NameError: name 'da_filled' is not defined

---
<a id="5-ttlsync"></a>
## 5  TTL Synchronisation (Real JPVA NPX + Bonsai)

This section demonstrates clock alignment using:
- NPX sync channel (`NPX/sync_channel.npy`)
- Bonsai HARP register stream from a behavior device

Set `NPX_FS_HZ` explicitly for your acquisition before running the TTL conversion cell.

In [ ]:
# ── 5a. Load real NPX sync and Bonsai TTL-like stream ────────────────────────
from data_conduit.ttlsync.ttlsync_core_pre import (
    extract_ttl_segments,
    align_pulse_tables,
    fit_linear_timebase,
    convert_timebase,
    TTLSyncModel,
)

# Required: set this argument to your real NPX sampling rate.
NPX_FS_HZ = None
if NPX_FS_HZ is None:
    raise ValueError("Set NPX_FS_HZ before running this cell.")

sync_candidates = [
    npx_directory_path / "sync_channel.npy",
    npx_directory_path / "sync_channel_1014.npy",
    misc_directory_path / "sync_channel_1014.npy",
]
sync_path = next((p for p in sync_candidates if p.exists()), None)
if sync_path is None:
    raise FileNotFoundError("Could not find NPX sync channel in expected locations")

npx_sync = np.load(sync_path)
npx_t = np.arange(len(npx_sync), dtype=float) / float(NPX_FS_HZ)

b0 = behavior_device.dfs_dict.get("Behavior0", {})
if "34" not in b0:
    raise KeyError("Behavior0 register '34' not found for Bonsai TTL source")
b34_df = b0["34"]

bonsai_t = b34_df.index.to_numpy(dtype=float)
if "Value" in b34_df.columns:
    bonsai_v = b34_df["Value"].to_numpy(dtype=float)
else:
    bonsai_v = b34_df.iloc[:, 0].to_numpy(dtype=float)

print(f"NPX sync file: {sync_path.name}, samples={len(npx_sync)}")
print(f"Bonsai Behavior0 register 34 samples={len(bonsai_v)}")

In [ ]:
# ── 5b. Extract pulse segments from real streams ─────────────────────────────
npx_thr = float(np.nanpercentile(npx_sync, 90))
bonsai_thr = float(np.nanpercentile(bonsai_v, 90))

npx_segments, npx_offset = extract_ttl_segments(
    npx_t, npx_sync, threshold=npx_thr, active_high=True, drop_inactive=True, align_to_zero=False,
 )
bonsai_segments, bonsai_offset = extract_ttl_segments(
    bonsai_t, bonsai_v, threshold=bonsai_thr, active_high=True, drop_inactive=True, align_to_zero=False,
 )

print(f"NPX pulses: {len(npx_segments)} (offset={npx_offset:.6f})")
print(f"Bonsai pulses: {len(bonsai_segments)} (offset={bonsai_offset:.6f})")
bonsai_segments.head()

In [ ]:
# ── 5c. Align pulse tables and fit linear timebase ───────────────────────────
ref_aligned, tgt_aligned = align_pulse_tables(
    bonsai_segments, npx_segments, normalise_start=False,
 )

fit_stats = fit_linear_timebase(ref_aligned, tgt_aligned, use="start")
print(f"Aligned pulse pairs: {len(ref_aligned)}")
print(f"slope={fit_stats['slope']:.9f}")
print(f"intercept={fit_stats['intercept']:.9f}")
print(f"r2={fit_stats['r2']:.9f}")

In [ ]:
# ── 5d. Convert NPX pulse starts into Bonsai timebase ───────────────────────
converted_npx = convert_timebase(
    tgt_aligned["Start"],
    slope=fit_stats["slope"],
    intercept=fit_stats["intercept"],
)

err = converted_npx - ref_aligned["Start"].to_numpy()
print(f"max abs error: {np.nanmax(np.abs(err)):.6e} s")
print(f"mean abs error: {np.nanmean(np.abs(err)):.6e} s")

In [ ]:
# ── 5e. TTLSyncModel convenience API ─────────────────────────────────────────
ttl_model = TTLSyncModel.fit(ref_aligned, tgt_aligned, use="start")
print(ttl_model)

test_npx_times = np.array([0.5, 5.0, 10.0])
print("NPX test times:", test_npx_times)
print("Mapped to Bonsai:", ttl_model.transform(test_npx_times))

---
<a id="6-globaltimes"></a>
## 6  Global Timebase Mapping (Real Nosepoke Array)

Now we build a global clock directly from the real nosepoke array time span and map one channel onto it.

In [ ]:
# ── 6a. Create global clock from real data span ──────────────────────────────
from data_conduit.globaltimes import create_global_clock, index_map_util

time_dim = "Time" if "Time" in da_filled.dims else da_filled.dims[0]
tvals = da_filled.coords[time_dim].to_numpy(dtype=float)

global_clock = create_global_clock(
    start_time=float(np.nanmin(tvals)),
    end_time=float(np.nanmax(tvals)),
    timestep_interval=0.01,  # 100 Hz grid
    include_end_time=True,
 )

print(f"Time dim: {time_dim}")
print(f"Global clock points: {len(global_clock)}")

In [ ]:
# ── 6b. Map a real channel stream to global clock ─────────────────────────────
global_dim = [d for d in da_filled.dims if d != time_dim][0]
first_channel = da_filled.coords[global_dim].values[0]
channel_series = da_filled.sel({global_dim: first_channel}).dropna(time_dim)
channel_t = channel_series.coords[time_dim].to_numpy(dtype=float)

result = index_map_util(global_clock, channel_t, match_type="nearest")

print("index_position_array shape:", result["index_position_array"].shape)
print("matched_global_times shape:", result["matched_global_times"].shape)
print("delta shape:", result["global_stream_time_delta"].shape)

print("First five mappings [global_idx, stream_idx]:")
print(result["index_position_array"][:5])

In [ ]:
# ── 6c. Compare matching strategies ───────────────────────────────────────────
for match in ["nearest", "before", "after", "exact"]:
    r = index_map_util(global_clock[:50], channel_t, match_type=match)
    idxs = r["index_position_array"][:10, 1]
    print(f"{match:>7}: {idxs.tolist()}")

---
<a id="7-segment"></a>
## 7  Segmentation Utilities (Real JPVA Channels)

We segment real channel activity from the loaded nosepoke arrays and also build event-centered windows around real ExperimentEvents timestamps.

In [ ]:
# ── 7a. segment_boolean_series on real channel activity ──────────────────────
from data_conduit.segment.segment_core_pre import (
    segment_boolean_series,
    slice_event_windows,
    slice_dataarray_windows,
)

global_dim = [d for d in da_filled.dims if d != time_dim][0]
ch0 = da_filled.coords[global_dim].values[0]
sig = da_filled.sel({global_dim: ch0})

state_series = pd.Series(
    (sig.fillna(0).to_numpy() > 0).astype(int),
    index=pd.Index(sig.coords[time_dim].to_numpy(dtype=float), name="Time"),
)

segments = segment_boolean_series(state_series)
print(f"Channel: {ch0}, segments found: {len(segments)}")
segments.head()

In [ ]:
# ── 7b. Keep active segments only with min duration ───────────────────────────
active_only = segment_boolean_series(
    state_series,
    drop_inactive=True,
    min_duration=0.05,
 )
print("Active-only segments:", len(active_only))
active_only.head()

In [ ]:
# ── 7c. slice_event_windows around real ExperimentEvents timestamps ──────────
events_folder = next(iter(events_src.dfs_dict))
events_file = next(iter(events_src.dfs_dict[events_folder]))
events_df = events_src.dfs_dict[events_folder][events_file]

event_times = events_df.index.to_numpy(dtype=float)[:3].tolist()
signal_df = pd.DataFrame({"Value": sig.to_numpy()}, index=pd.Index(sig.coords[time_dim].to_numpy(dtype=float), name="Time"))

windows = slice_event_windows(
    signal_df,
    event_times=event_times,
    pre=1.0,
    post=1.0,
)

print(f"Event windows extracted: {len(windows)}")
for ev, win in windows.items():
    print(f"event={ev:.3f}, rows={len(win)}")

In [ ]:
# ── 7d. slice_dataarray_windows on real DataArray ─────────────────────────────
da_event_times = event_times if len(event_times) else [float(sig.coords[time_dim].to_numpy()[0])]
da_windows = slice_dataarray_windows(
    da_filled,
    event_times=da_event_times,
    pre=1.0,
    post=1.0,
    time_dim=time_dim,
)

print(f"DataArray windows: {len(da_windows)}")
for ev, win_da in da_windows.items():
    print(f"event={ev:.3f}, shape={win_da.shape}")

---
<a id="8-utilities"></a>
## 8  Timestamps & Utilities (Real JPVA Objects)

Final section: run timestamp and utility helpers on already-loaded real DataFrames/DataArrays.

In [ ]:
# ── 8a. collect_timestamps on real DataFrame ─────────────────────────────────
from data_conduit.timestamps import collect_timestamps, collect_timestamps_dict

real_df = b34_df
ts = collect_timestamps(real_df)
print(f"Timestamps count: {len(ts)}")
print(f"Range: [{float(np.min(ts)):.6f}, {float(np.max(ts)):.6f}]")

df_with_time_col = real_df.reset_index().rename(columns={real_df.index.name or "index": "Time"})
ts_col = collect_timestamps(df_with_time_col, time_location="columns")
print("From Time column count:", len(ts_col))

In [ ]:
# ── 8b. collect_timestamps_dict on real register dict ────────────────────────
b0_dict = behavior_device.dfs_dict.get("Behavior0", {})

all_ts = collect_timestamps_dict(b0_dict, return_type="list", verbose=False)
print(f"Behavior0 all unique timestamps: {len(all_ts)}")

ts_dict = collect_timestamps_dict(b0_dict, return_type="dict", verbose=False)
counts = {k: len(v) for k, v in ts_dict.items()}
print("Per-register timestamp counts:")
print(counts)

In [ ]:
# ── 8c. Utilities on real and toy names ──────────────────────────────────────
from data_conduit.utils import starts_with, ends_with, contains
from data_conduit.utils import _flatten_nested_dict, _get_nested_dict_depth

print("starts_with('Behavior')('Behavior3'):", starts_with("Behavior")("Behavior3"))
print("ends_with('Rotation')('InnerRotation'):", ends_with("Rotation")("InnerRotation"))
print("contains('Events')('ExperimentEvents'):", contains("Events")("ExperimentEvents"))

nested_preview = {
    "Behavior0": {"34": "...", "35": "..."},
    "ExperimentEvents": {"session": "..."},
}
print("Nested dict depth:", _get_nested_dict_depth(nested_preview))
print("Flattened:", _flatten_nested_dict(nested_preview, separator="/"))

---
<a id="9-api"></a>
## 9  API Quick-Reference

### IO (`data_conduit.io`)
| Function | Purpose |
|----------|---------|
| `collect_folders(base_path, folder_prefix)` | List subdirectories with optional prefix filter |
| `collect_dfs(base_path, readers, ...)` | **Master function:** directory → nested dict of DataFrames |
| `read_csv(path, **kwargs)` | Read CSV with sensible defaults |
| `read_json(path, **kwargs)` | Read JSON/JSONL |
| `split_jsonl(path)` | Split JSONL → (metadata_df, trials_df) |
| `split_yaml(path)` | Split YAML → (metadata_df, trials_df) |
| `add_reader(name, fn)` | Register a custom reader |
| `get_reader(name)` | Retrieve a registered reader |

### DataSource (`data_conduit.datasource`)
| Class | Purpose |
|-------|---------|
| `DataSource(experiment_directory_path, readers, datasource_data_arrays, ...)` | Base: directory → dfs_dict + named DataArrays |
| `Device(experiment_directory_path, harp_device_yaml_path, device_type, ...)` | HARP .bin device wrapper |
| `SoundCard(...)` | Preset: SoundCard registers 8, 32, 33, 35 |
| `CameraStart(...)` | Preset: camera start register 78 |
| `Camera0Frames(...)` | Preset: camera frames with optional matching filter |
| `AnalogSync(...)` | Preset: analog sync with optional matching filter |
| `FileTypeData(experiment_directory_path, file_type, ...)` | Flat file (CSV/JSON/YAML) loader |
| `ExperimentEvents(...)` | Preset: CSV events with Value→Event rename |
| `RotationData(...)` | Preset: rotation encoder CSV with angular unit conversion |
| `VideoData(...)` | Preset: VideoData CSV |
| `VisualEnvironment(...)` | Preset: headerless visual environment CSV |
| `RingDebugData(...)` | Preset: YAML ring debug split |

### MultiSource (`data_conduit.multisource`)
| Class | Purpose |
|-------|---------|
| `MultiSource(dfs_dict, virtual_maps, ...)` | Base: dfs_dict + virtual maps → data_arrays + lookup_arrays |
| `MultiDevice(experiment_directory_path, ...)` | HARP device loading + auto virtual-map building |
| `Nosepoke(...)` | Preset: 6 devices × 3 local IDs for nosepoke peripherals |

### Virtual Arrays (`data_conduit.virtualarrays`)
| Function/Class | Purpose |
|----------------|---------|
| `construct_data_array(virtual_map, times, ...)` | Build base xarray DataArray with virtual coords |
| `construct_lookup_array(virtual_map, ...)` | Build 1D lookup array |
| `update_data_array(data_array, virtual_map, dfs_dict, ...)` | Populate DataArray with values from dfs_dict |
| `ulookup(lookup_array, **selectors)` | Query: find global coords by virtual coord criteria |
| `LookupAccessorConstructor` | xarray `.ulookup()` accessor (auto-constructs lookup) |

### TTL Sync (`data_conduit.ttlsync`)
| Function/Class | Purpose |
|----------------|---------|
| `extract_ttl_segments(times, values, ...)` | Binarize waveform → pulse table |
| `align_pulse_tables(ref_df, tgt_df, ...)` | Align two pulse tables for fitting |
| `fit_linear_timebase(ref_df, tgt_df, ...)` | Fit linear target→reference model |
| `convert_timebase(values, slope, intercept)` | Apply linear transform |
| `get_ttl_timebase_conversion(ref, tgt, ...)` | End-to-end conversion convenience |
| `get_npx_to_bonsai_time_conversion(...)` | NPX/Bonsai semantic wrapper |
| `TTLSyncModel(slope, intercept, r2)` | Dataclass with `.fit()` and `.transform()` |

### Global Times (`data_conduit.globaltimes`)
| Function | Purpose |
|----------|---------|
| `create_global_clock(start, end, step)` | Evenly spaced canonical time vector |
| `index_map_util(global_clock, stream_times, match_type)` | Map stream → global clock (nearest/before/after/exact) |

### Timestamps (`data_conduit.timestamps`)
| Function | Purpose |
|----------|---------|
| `collect_timestamps(df, ...)` | Extract timestamps from one DataFrame |
| `collect_timestamps_dict(dfs, ...)` | Extract from flat dict of DataFrames |
| `collect_timestamps_nested(dfs_dict, ...)` | Extract from arbitrarily nested dict |

### Segmentation (`data_conduit.segment`)
| Function | Purpose |
|----------|---------|
| `segment_boolean_series(series, ...)` | Run-length segmentation of boolean series |
| `slice_event_windows(df, event_times, pre, post)` | Cut DataFrame into event-centered windows |
| `slice_dataarray_windows(da, event_times, pre, post)` | Cut DataArray into event-centered windows |

### Utils (`data_conduit.utils`)
| Function | Purpose |
|----------|---------|
| `starts_with(prefix)` | Selector callable: key starts with prefix |
| `ends_with(suffix)` | Selector callable: key ends with suffix |
| `contains(substring)` | Selector callable: key contains substring |

### HarpTools (`data_conduit.harptools`)
| Function | Purpose |
|----------|---------|
| `read_harp_bin(path, harp_device_yaml_path)` | Read HARP .bin register file → DataFrame |
| `construct_device_reader(yaml_path)` | Build HARP reader from YAML schema |
| `collect_registers(yaml_path, addresses)` | Look up register names by address |

In [ ]:
# ── Cleanup temporary directory ──────────────────────────────────────────────
shutil.rmtree(TMPDIR, ignore_errors=True)
print(f"Cleaned up {TMPDIR}")